[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/idc-monai/blob/main/monai_contribution/idc_dataset.ipynb)

# Using NCI Imaging Data Commons with MONAI

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial demonstrates how to use imaging data from the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) with MONAI for medical imaging AI tasks.

**IDC** is a cloud-based repository of publicly available cancer imaging data with:
- ~100 TB of radiology and pathology images
- No authentication required for access
- AI-generated and expert annotations
- All data in DICOM format

## Setup environment

In [ ]:
!pip install -q monai idc-index itk itkwasm-dicom

# Restart runtime after installing ITK (required for ITK to load properly)
import sys
if "google.colab" in sys.modules:
    try:
        import itk
    except ImportError:
        print("Restarting runtime to load ITK...")
        import os
        os.kill(os.getpid(), 9)

## Setup imports

In [ ]:
import os
import tempfile
from pathlib import Path
from typing import Hashable, Mapping

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from idc_index import IDCClient

from monai.config import KeysCollection
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    MapTransform,
)
from monai.data import Dataset, DataLoader, MetaTensor
from monai.data.image_reader import ITKReader
import monai

monai.config.print_config()

## 1. Query IDC Data

Use `idc-index` to query IDC's metadata index using SQL.

In [ ]:
# Initialize IDC client
client = IDCClient()

# Check IDC version and data scale
print(f"IDC version: {client.get_idc_version()}")

stats = client.sql_query("""
    SELECT COUNT(DISTINCT collection_id) as collections,
           COUNT(DISTINCT PatientID) as patients
    FROM index
""")
print(f"Collections: {stats.iloc[0]['collections']}, Patients: {stats.iloc[0]['patients']}")

In [ ]:
# Find lung CT collections by joining collections_index with index
client.fetch_index("collections_index")

# Join to get modality information (not available in collections_index directly)
lung_collections = client.sql_query("""
    SELECT c.collection_id, c.Subjects, c.CancerTypes,
           COUNT(DISTINCT CASE WHEN i.Modality = 'CT' THEN i.SeriesInstanceUID END) as ct_series
    FROM collections_index c
    JOIN index i ON c.collection_id = i.collection_id
    WHERE c.CancerTypes LIKE '%Lung%'
    GROUP BY c.collection_id, c.Subjects, c.CancerTypes
    HAVING ct_series > 0
    ORDER BY c.Subjects DESC
    LIMIT 5
""")
print("Lung CT collections:")
print(lung_collections.to_string(index=False))

In [ ]:
# Query specific CT series (using small rider_pilot collection for demo)
series_df = client.sql_query("""
    SELECT SeriesInstanceUID, PatientID, Modality,
           ROUND(series_size_MB, 2) as size_mb
    FROM index
    WHERE collection_id = 'rider_pilot' AND Modality = 'CT'
    LIMIT 3
""")
print(f"Found {len(series_df)} CT series")

## 2. Download DICOM Data

In [ ]:
# Download to temporary directory
data_dir = tempfile.mkdtemp(prefix="idc_monai_")
series_uids = list(series_df['SeriesInstanceUID'])

print(f"Downloading {len(series_uids)} series...")
client.download_from_selection(
    seriesInstanceUID=series_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"
)
print("Done!")

## 3. Load with MONAI Transforms

MONAI's `LoadImaged` with `ITKReader` directly loads DICOM series from directories.

In [ ]:
# Define transforms for CT preprocessing
# Use ITKReader explicitly to load DICOM series from directories
transforms = Compose([
    LoadImaged(keys=["image"], reader=ITKReader()),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 2.0)),
    ScaleIntensityRanged(keys=["image"], a_min=-175, a_max=250,
                         b_min=0.0, b_max=1.0, clip=True),
])

# Create dataset
data_dicts = [{"image": os.path.join(data_dir, uid)} for uid in series_uids]
dataset = Dataset(data=data_dicts, transform=transforms)

# Load sample
sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Value range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")

## 4. Visualize

In [ ]:
image = sample['image'][0]
z = image.shape[2] // 2

plt.figure(figsize=(6, 6))
plt.imshow(image[:, :, z].T, cmap='gray', origin='lower')
plt.title(f'CT from IDC (slice {z})')
plt.axis('off')
plt.show()

## 5. Loading DICOM Segmentations

IDC contains DICOM Segmentation (DICOM-SEG) objects. These are different from regular DICOM series:
- **Enhanced multiframe**: All slices in a single file
- **Cannot use ITKReader**: Standard readers don't support this format
- **Segment metadata**: Each segment has label, description, and recommended display color

We define a custom `LoadDicomSegd` transform that:
1. Uses `itkwasm-dicom` (wraps dcmqi) for robust DICOM-SEG reading
2. Automatically transforms the data to match MONAI's ITKReader output
3. Ensures CT and SEG have matching affines for direct overlay
4. Extracts segment colors from DICOM metadata for visualization

In [ ]:
# Fetch seg_index to find paired image-segmentation data
client.fetch_index("seg_index")

# Find CT with TotalSegmentator segmentations (104 anatomical structures)
paired = client.sql_query("""
    SELECT src.SeriesInstanceUID as image_uid,
           seg.SeriesInstanceUID as seg_uid,
           src.collection_id, seg.total_segments,
           ROUND(src.series_size_MB, 2) as image_mb
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    WHERE src.Modality = 'CT'
      AND seg.AlgorithmName LIKE '%TotalSegmentator%'
    ORDER BY src.series_size_MB ASC
    LIMIT 3
""")
print("CT with TotalSegmentator segmentations:")
print(paired.to_string(index=False))

In [ ]:
# Download one image-segmentation pair
demo_pair = paired.iloc[0]
seg_dir = tempfile.mkdtemp(prefix="idc_seg_")

print(f"Downloading image and segmentation pair...")
client.download_from_selection(
    seriesInstanceUID=[demo_pair['image_uid'], demo_pair['seg_uid']],
    downloadDir=seg_dir,
    dirTemplate="%SeriesInstanceUID"
)
print("Done!")

In [ ]:
import itkwasm_dicom

class LoadDicomSegd(MapTransform):
    """Load DICOM Segmentation (DICOM-SEG) files using ITKWasm.
    
    DICOM-SEG is an enhanced multiframe DICOM format that stores segmentation
    masks with segment metadata including recommended display colors.
    """
    
    def __init__(self, keys: KeysCollection, allow_missing_keys: bool = False):
        super().__init__(keys, allow_missing_keys)
    
    def _find_dcm_file(self, path: Path) -> Path:
        """Find .dcm file in directory or return path if already a file."""
        if path.is_file():
            return path
        dcm_files = list(path.glob("*.dcm"))
        if not dcm_files:
            raise FileNotFoundError(f"No .dcm files found in {path}")
        return dcm_files[0]
    
    def _build_affine(self, spacing, origin, direction, size):
        """Build 4x4 affine matrix compatible with ITKReader/MONAI conventions."""
        flip_axes = []
        new_origin = origin.copy()
        
        # Y axis: flip if direction is negative
        if direction[1, 1] < 0:
            flip_axes.append(1)
            new_origin[1] = origin[1] + (size[1] - 1) * spacing[1] * direction[1, 1]
        new_origin[1] = -new_origin[1]
        
        # Z axis: flip if direction is negative
        if direction[2, 2] < 0:
            flip_axes.append(2)
            new_origin[2] = origin[2] + (size[2] - 1) * spacing[2] * direction[2, 2]
        
        # X axis: negate origin
        new_origin[0] = -origin[0]
        
        # Build affine with ITKReader-like diagonal
        affine = np.eye(4)
        affine[:3, :3] = np.diag([-spacing[0], -spacing[1], spacing[2]])
        affine[:3, 3] = new_origin
        
        return affine, flip_axes
    
    def __call__(self, data: Mapping[Hashable, any]) -> dict[Hashable, any]:
        d = dict(data)
        for key in self.key_iterator(d):
            path = Path(d[key])
            dcm_file = self._find_dcm_file(path)
            
            # Read using ITKWasm
            seg_image, overlay_info = itkwasm_dicom.read_segmentation(dcm_file)
            
            # Transpose array from (Z, Y, X) to (X, Y, Z)
            seg_array = np.asarray(seg_image.data).copy()
            seg_array = np.transpose(seg_array, (2, 1, 0))
            
            # Build affine from spatial metadata
            spacing = np.array(seg_image.spacing)
            origin = np.array(seg_image.origin)
            direction = np.array(seg_image.direction).reshape(3, 3)
            size = np.array(seg_image.size)
            
            affine, flip_axes = self._build_affine(spacing, origin, direction, size)
            
            # Flip array axes as needed
            for axis in flip_axes:
                seg_array = np.flip(seg_array, axis=axis)
            seg_array = np.ascontiguousarray(seg_array)
            
            # Create MONAI MetaTensor with metadata
            meta_tensor = MetaTensor(seg_array)
            meta_tensor.affine = affine
            meta_tensor.meta["filename_or_obj"] = str(dcm_file)
            meta_tensor.meta["overlay_info"] = overlay_info
            meta_tensor.meta["original_channel_dim"] = "no_channel"
            
            d[key] = meta_tensor
            d[f"{key}_meta_dict"] = dict(meta_tensor.meta)
        
        return d

# Load CT with MONAI's ITKReader
ct_transforms = Compose([
    LoadImaged(keys=["image"], reader=ITKReader()),
    EnsureChannelFirstd(keys=["image"]),
])

# Load SEG with our custom LoadDicomSegd
seg_transforms = Compose([
    LoadDicomSegd(keys=["label"]),
    EnsureChannelFirstd(keys=["label"]),
])

# Load both
image_path = os.path.join(seg_dir, demo_pair['image_uid'])
seg_path = os.path.join(seg_dir, demo_pair['seg_uid'])

ct_data = ct_transforms({"image": image_path})
seg_data = seg_transforms({"label": seg_path})

ct_image = ct_data["image"]
seg_label = seg_data["label"]

print(f"CT image shape: {ct_image.shape}")
print(f"Segmentation shape: {seg_label.shape}")
print(f"Unique labels: {torch.unique(seg_label)[:10].tolist()}...")  # First 10 labels

In [ ]:
# Verify alignment - affines should match!
print("CT affine:")
print(ct_image.affine)
print("\nSEG affine:")
print(seg_label.affine)

# Check that they match (LoadDicomSegd produces ITKReader-compatible affines)
affine_match = torch.allclose(ct_image.affine, seg_label.affine, atol=1e-4)
print(f"\nAffines match: {affine_match}")

if affine_match:
    print("✓ CT and SEG are spatially aligned - voxel indices correspond directly!")

## 5.1 Exploring Segment Metadata

DICOM SEG files contain rich metadata about each segment, including:
- **SegmentLabel**: Human-readable name (e.g., "Liver", "Spleen")
- **SegmentedPropertyCategoryCodeSequence**: Category (e.g., "Anatomical Structure", "Tumor")
- **SegmentedPropertyTypeCodeSequence**: Specific type using standardized codes (e.g., SNOMED-CT)
- **recommendedDisplayRGBValue**: Display color for visualization
- **labelID**: Integer label value in the segmentation array

In [ ]:
def get_segment_info(overlay_info):
    """Extract segment information from DICOM SEG overlay_info.
    
    Args:
        overlay_info: Dictionary from LoadDicomSegd containing segmentAttributes
        
    Returns:
        List of dicts with segment details (label_id, name, category, type, color)
    """
    segment_attrs = overlay_info.get("segmentAttributes", [[]])
    
    # Flatten segment attributes (may be nested in groups)
    all_segments = []
    for group in segment_attrs:
        all_segments.extend(group)
    
    segments = []
    for seg in all_segments:
        # Extract category (e.g., "Anatomical Structure", "Morphologically Altered Structure")
        category_seq = seg.get("SegmentedPropertyCategoryCodeSequence", {})
        category = category_seq.get("CodeMeaning", "Unknown")
        
        # Extract type (e.g., "Liver", "Spleen") with coding scheme
        type_seq = seg.get("SegmentedPropertyTypeCodeSequence", {})
        seg_type = type_seq.get("CodeMeaning", "Unknown")
        type_code = type_seq.get("CodeValue", "")
        coding_scheme = type_seq.get("CodingSchemeDesignator", "")
        
        # Extract modifier if present (e.g., "Left", "Right")
        modifier_seq = seg.get("SegmentedPropertyTypeModifierCodeSequence", {})
        modifier = modifier_seq.get("CodeMeaning", "") if modifier_seq else ""
        
        segments.append({
            "label_id": seg.get("labelID", 0),
            "name": seg.get("SegmentLabel", "Unknown"),
            "category": category,
            "type": seg_type,
            "type_code": f"{coding_scheme}:{type_code}" if type_code else "",
            "modifier": modifier,
            "color_rgb": seg.get("recommendedDisplayRGBValue", [128, 128, 128]),
            "algorithm": seg.get("SegmentAlgorithmName", ""),
        })
    
    return sorted(segments, key=lambda x: x["label_id"])

# Extract segment information
overlay_info = seg_data.get('label_meta_dict', {}).get('overlay_info', {})
segments = get_segment_info(overlay_info)

print(f"Found {len(segments)} segments in DICOM SEG:\n")
print(f"{'ID':<4} {'Name':<25} {'Category':<25} {'Type':<20} {'Code'}")
print("-" * 100)
for seg in segments[:15]:  # Show first 15
    print(f"{seg['label_id']:<4} {seg['name']:<25} {seg['category']:<25} {seg['type']:<20} {seg['type_code']}")

if len(segments) > 15:
    print(f"... and {len(segments) - 15} more segments")

In [ ]:
# Create a lookup dictionary: label_id -> segment name
# Useful for interpreting model predictions or filtering specific structures
label_to_name = {seg["label_id"]: seg["name"] for seg in segments}

# Prepare arrays for visualization
ct_np = ct_image[0].numpy()  # Remove channel dim
seg_np = seg_label[0].numpy()

# Find a slice with segmentation labels
z_slices_with_labels = np.where(seg_np.sum(axis=(0, 1)) > 0)[0]
if len(z_slices_with_labels) > 0:
    z_mid = z_slices_with_labels[len(z_slices_with_labels) // 2]
else:
    z_mid = ct_np.shape[2] // 2

# Example: Find which segments are present in the current slice
seg_slice = seg_np[:, :, z_mid]
labels_in_slice = [int(l) for l in np.unique(seg_slice) if l > 0]

print(f"Segments visible in slice {z_mid}:")
for label_id in labels_in_slice[:10]:
    seg_info = next((s for s in segments if s["label_id"] == label_id), None)
    if seg_info:
        color = seg_info["color_rgb"]
        print(f"  Label {label_id}: {seg_info['name']:<20} (RGB: {color})")

if len(labels_in_slice) > 10:
    print(f"  ... and {len(labels_in_slice) - 10} more")

In [ ]:
def build_seg_colormap(overlay_info):
    """Build a matplotlib colormap from DICOM SEG segment colors.
    
    Args:
        overlay_info: Dictionary from LoadDicomSegd containing segmentAttributes
        
    Returns:
        ListedColormap with colors for each segment label
    """
    segment_attrs = overlay_info.get("segmentAttributes", [[]])
    
    # Flatten segment attributes (may be nested in groups)
    all_segments = []
    for group in segment_attrs:
        all_segments.extend(group)
    
    if not all_segments:
        return plt.cm.nipy_spectral
    
    max_label = max(seg.get("labelID", 0) for seg in all_segments)
    
    # Build RGBA color array: index 0 = background (transparent)
    colors = np.zeros((max_label + 1, 4))
    colors[0] = [0, 0, 0, 0]  # Background transparent
    
    for seg in all_segments:
        label_id = seg.get("labelID", 0)
        rgb = seg.get("recommendedDisplayRGBValue", [128, 128, 128])
        colors[label_id] = [rgb[0]/255, rgb[1]/255, rgb[2]/255, 1.0]
    
    return ListedColormap(colors)

# Build colormap from DICOM SEG metadata
seg_cmap = build_seg_colormap(overlay_info)

# Visualize - no reorientation needed since affines already match!
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CT image only
axes[0].imshow(ct_np[:, :, z_mid].T, cmap='gray', origin='lower', vmin=-1000, vmax=500)
axes[0].set_title('CT Image')
axes[0].axis('off')

# Segmentation only (using DICOM SEG colors)
axes[1].imshow(seg_np[:, :, z_mid].T, cmap=seg_cmap, origin='lower',
               vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
axes[1].set_title(f'Segmentation ({int(seg_np.max())} labels)\n(DICOM SEG colors)')
axes[1].axis('off')

# Overlay - direct overlay works because affines match!
axes[2].imshow(ct_np[:, :, z_mid].T, cmap='gray', origin='lower', vmin=-1000, vmax=500)
seg_slice = seg_np[:, :, z_mid]
mask = np.ma.masked_where(seg_slice == 0, seg_slice)
axes[2].imshow(mask.T, cmap=seg_cmap, alpha=0.6, origin='lower',
               vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
axes[2].set_title('Overlay\n(DICOM SEG colors)')
axes[2].axis('off')

plt.suptitle(f'CT + TotalSegmentator Segmentation (slice {z_mid})', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n✓ Segmentation loaded and aligned using LoadDicomSegd!")
print(f"  Colors extracted from DICOM SEG recommendedDisplayRGBValue.")

## 6. Check Licenses

In [ ]:
# Always check licenses before use
uid_list = ", ".join(f"'{uid}'" for uid in series_uids)
licenses = client.sql_query(f"""
    SELECT license_short_name, COUNT(*) as count
    FROM index WHERE SeriesInstanceUID IN ({uid_list})
    GROUP BY license_short_name
""")
print("Licenses:")
print(licenses.to_string(index=False))

## 7. Cleanup

In [ ]:
# import shutil; shutil.rmtree(data_dir)  # Uncomment to delete
print(f"Data at: {data_dir}")

## Summary

This tutorial demonstrated:
1. Querying IDC with `idc-index` SQL interface
2. Downloading DICOM data (no authentication needed)
3. Loading CT images into MONAI with `LoadImaged` and `ITKReader`
4. Finding paired segmentations via `seg_index`
5. Loading DICOM-SEG files with a custom `LoadDicomSegd` transform
6. Extracting segment metadata (names, categories, types, standardized codes)
7. Using DICOM SEG colors for visualization

**Key Point**: DICOM-SEG contains rich metadata. Each segment includes:
- `SegmentLabel` - Human-readable name
- `SegmentedPropertyCategoryCodeSequence` - Category (e.g., "Anatomical Structure")
- `SegmentedPropertyTypeCodeSequence` - Type with standardized codes (SNOMED-CT, etc.)
- `recommendedDisplayRGBValue` - Display color

**Resources:**
- [IDC Portal](https://portal.imaging.datacommons.cancer.gov/)
- [IDC Documentation](https://learn.canceridc.dev/)
- [idc-index](https://github.com/ImagingDataCommons/idc-index)
- [DICOM SEG Standard](https://dicom.nema.org/medical/dicom/current/output/chtml/part03/sect_C.8.20.html)
- [ITKWasm DICOM](https://wasm.itk.org/en/latest/introduction/file_formats/dicom.html)